## In this exercise, you'll put together a RAG system and compare outputs from RAG vs. just querying an LLM.

## For this exercise, you'll be asking about Subspace-Constrained LoRA (SC-LoRA), a new technique described in a [recent article publised on arXiv.org](https://arxiv.org/abs/2505.23724). You've been provided the text of this article in the file 2505.23724v1.txt.

In [1]:
# !pip install faiss-cpu
# !pip install langchain
# !pip install langchain-community
# !pip install langchain-openai
# !pip install langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is in

In [2]:
import os
import json

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from openai import OpenAI

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

/tmp/ipykernel_16030/2946999983.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## First, you'll need to set up a way to interact with the generator model. You can use the OpenAI class from the openai library for this. See [this page](https://developers.openai.com/api/docs/guides/text?lang=python) for more information. When you do this, you'll need to set the base_url to "https://openrouter.ai/api/v1" and to pass in your api key. Set the model to "openrouter/owl-alpha".

In [78]:
# api_key = ""

In [3]:
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

## First, ask the model "How does SC-LoRA differ from regular LoRA?" without providing any additional context. Read through a few different responses. Question: Are the responses accurate, or does it seem like the model is just making up something that sounds plausible?

In [30]:
model = "openrouter/owl-alpha"
query = "How does SC-LoRA differ from regular LoRA?"

In [4]:
response = client.responses.create(
    model=model,
    input=query
)

In [16]:
print(response.output_text)

SC-LoRA (Scaled and Calibrated LoRA) differs from regular LoRA (Low-Rank Adaptation) in several key ways, primarily focusing on improving the stability, performance, and adaptability of the adaptation process. Here are the main differences:

1.  **Initialization and Scaling:**
    *   **Regular LoRA:** Typically initializes the low-rank matrices $A$ and $B$ with random values (e.g., Gaussian for $A$, zeros for $B$) and applies a fixed scaling factor $\alpha/r$ to the output of the LoRA module.
    *   **SC-LoRA:** Introduces a more sophisticated initialization and scaling strategy. It often involves scaling the LoRA updates based on the magnitude of the pre-trained weights or the singular values of the weight matrices. This helps in maintaining a more stable training process and prevents the LoRA updates from being too large or too small relative to the pre-trained weights, which can lead to instability or slow convergence.

2.  **Calibration Mechanism:**
    *   **Regular LoRA:** Does

# Part 1: Manual RAG
In order to get more reliable responses, let's set up a RAG system.

In this first part, you'll build all of the pieces of the RAG system individually.

First, you'll need the retriever portion. Create a FAISS index to hold the text of the article. Encode this text using the all-MiniLM-L6-v2 encoder. Note that you'll want to divide the text into smaller chunks rather than encoding the whole article all at once. You could try, for example, the [RecursiveCharacterTextSplitter class from LangChain](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter). You'll need to specify a chunk_size and chunk_overlap. You could try a chunk_size of 500 and overlap of 50 as a starting point.

In [17]:
with open("2505.23724v1.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [21]:
lines = [line.strip() for line in text.split('\n') if len(line.strip()) > 10]
text = "\n".join(lines)
print(text[:1000])

arXiv:2505.23724v1  [cs.LG]  29 May 2025SC-LoRA: Balancing Efficient Fine-tuning and Knowledge Preservation via
Subspace-Constrained LoRA
Minrui Luo∗1,2, Fuhang Kuang∗1, Yu Wang3, Zirui Liu1, Tianxing He†1,2
1Institute for Interdisciplinary Information Sciences, Tsinghua University
2Shanghai Qi Zhi Institute
3Institute of Information Engineering, Chinese Academy of Sciences
{luomr22,kfh22,liu-zr22}@mails.tsinghua.edu.cn;
wangyu2002@iie.ac.cn hetianxing@mail.tsinghua.edu.cn
Parameter-Efficient Fine-Tuning (PEFT)
methods, particularly Low-Rank Adaptation
(LoRA), are indispensable for efficiently
customizing Large Language Models (LLMs).
However, vanilla LoRA suffers from slow
convergence speed and knowledge forgetting
problems. Recent studies have leveraged
the power of designed LoRA initialization,
to enhance the fine-tuning efficiency, or to
preserve knowledge in the pre-trained LLM.
However, none of these works can address
the two cases at the same time. To this end,
we introduce Subs

In [27]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

text_list = text_splitter.split_text(text)

In [72]:
embeddings_model = "all-MiniLM-L6-v2"

In [ ]:
transformer = SentenceTransformer(embeddings_model)

In [31]:
query_embedding = transformer.encode(query)
text_embeddings = transformer.encode(text_list)

In [32]:
index = faiss.IndexFlatIP(text_embeddings.shape[1]) # build the index
print(index.is_trained)
index.add(text_embeddings) # add vectors to the index
print(index.ntotal)

True
55


Next, use the following as a system prompt:

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)

```

Use the FAISS index to pull in relevant context to fill in the context. Try passing in this additional system prompt. Hint: you can do this by using the following messages in the client.chat.completions.create function

```
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
```

How does adding this context change the results?

In [35]:
k = 5
D, I = index.search(np.array([query_embedding]), k)

In [61]:
context = ""
for i in I[0]:
    context = context + text_list[i]

In [66]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
  )

In [67]:
messages=[
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": query
    }
]

In [68]:
response = client.responses.create(
    model=model,
    input=messages
)

In [70]:
response.output_text

'Based on the provided context, SC-LoRA differs from regular LoRA primarily through its initialization and its focus on preserving existing knowledge. While regular LoRA initializes adapter matrices with Gaussian noise and zero, SC-LoRA uses a specific initialization involving world knowledge samples to prevent "catastrophic forgetting." Additionally, SC-LoRA introduces a parameter ($\\beta$) that allows for a balance between fine-tuning performance and the preservation of safety and world knowledge.'

# Part 2: LangChain
You can also use the [LangChain library](https://www.langchain.com/) to help build your RAG system.

For the retriever, you can use the [HugginFaceEmbeddings class](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), using the all-MiniLM-L6-v2 model, to create your embedding model. There is also a [FAISS class](https://reference.langchain.com/python/langchain-community/vectorstores/faiss/FAISS), which has a useful from_texts method. Once you've created your vector store, use the [as_retriever method](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.as_retriever) on it and save it to a variable named retriever.

In [73]:
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
hf_embedder = HuggingFaceEmbeddings(
    model_name=embeddings_model,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [74]:
vector_store = FAISS.from_texts(text_list, hf_embedder)

In [76]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 50}
)

For the generator, you can use the [ChatOpenAI class](https://python.langchain.com/docs/integrations/chat/openai/). Be sure to set base_url="https://openrouter.ai/api/v1", model_name="openrouter/owl-alpha", and openai_api_key= Your API key. Save this to a variable named llm.

In [79]:
llm = ChatOpenAI(
    openai_api_key=api_key,
    model_name=model,
    base_url="https://openrouter.ai/api/v1"
)

Now that the two components have been created, we can combine them into a chat template using the [ChatPromptTemplate class](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html). We can set up a system prompt and the pass that in, like

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
```

In [83]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

Now we need to connect all of the pieces together. Newer versions of LangChain commonly use LCEL (LangChain Expression Language)[https://www.langchain.com/blog/langchain-expression-language] to build pipelines where components are connected together using the | operator.

You'll need:

A helper function that combines retrieved documents into a single string of context
A pipeline that:
extracts the question from the input
sends the question to the retriever
formats the retrieved documents into context
passes both the context and question into the prompt
sends the completed prompt to the LLM
A simplified diagram looks like this:

input ↓ extract question ↓ retrieve documents ↓ format context ↓ prompt ↓ LLM

You can create this chain by using

```
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)
```
Take a minute to study this and see if you can figure out how the syntax works.

In [84]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)

Finally, invoke your chain using:

```
response = rag_chain.invoke(
    {"question": query}
)
```

print(response.content)
Compare the output from this section with both previous approaches:

LLM without retrieval
Manual RAG
LangChain RAG
The quality of the answers from (2) and (3) should be similar. The purpose of LangChain is not to improve the model itself. Rather, it provides abstractions that simplify the retrieval and generation workflow.

In [85]:
response = rag_chain.invoke(
    {"question": query}
)

In [87]:
response.content

'SC-LoRA differs from regular LoRA by using a subspace-constrained initialization method that preserves world knowledge and improves fine-tuning performance. While regular LoRA initializes the down-projection matrix with Gaussian noise and the up-projection matrix with zeros, SC-LoRA optimizes the initialization to maximize context information retention and minimize knowledge forgetting. This approach allows SC-LoRA to achieve better utility, even surpassing full fine-tuning in some cases, while maintaining safety alignment.'